# Görev F — Eşleşmiş sonda: eğitim/eval uyumsuzluğu testi

**§19'un çelişkisi:** Eğitim cross-chunk görevini **öğrendi** (lossB 3.45 → 1.51-2.23,
K=14 ≈ 3600 token) ama aynı mesafede **eval şansta** kaldı. Öğrenilmiş bir beceri
değerlendirmede görünmüyorsa, ikisi aynı görevi ölçmüyordur.

**Tespit edilen fark:** Eğitimde hedef, yoğun bir chunk'ın **sonuna** (sınıra hizalı)
yazılıyor ve sorgu yine yoğun chunk'ın sonunda soruluyor. Eski eval'de hedef **boş
akışın ilk iki token'ı**, sorgu **çıplak bir key**. Yani mesafe değil, **sahne** değişiyordu.

**Bu test:** Sondayı eğitim dağılımından üretir (A dense+hedef sonda → K dolgu → B
dense+sorgu sonda); tek değişken K. **Aynı koşuda eski sondayı da ölçer**, fark
doğrudan yan yana görülür. Eğitim YOK — Görev E checkpoint'leri kullanılır (~10 dk).

**ÖN-KAYITLI KRİTERLER:**
- **UYUMSUZLUK DOĞRULANDI:** K ≤ 16'da eşleşmiş ≥ %15 iken eski ≤ %8 →
  §17-§19 çöküşü büyük ölçüde **eval artefaktı**; dürüst iddia:
  *"eğitildiği mesafede, eğitildiği biçimde sorulduğunda hatırlar"*.
- **MİMARİ ŞÜPHESİ:** eşleşmiş sondada da K=8/16'da < %8 → dört eleme tamamlanır
  (yasa/kapasite/müfredat/harness); baş şüpheli state boyutu (bulk_dim=32) + okuma yolu.
- **EKSTRAPOLASYON:** K=32/64 (eğitilen tavanın 2-4 katı) ayrıca raporlanır.

GPU **T4**, Internet **On** → Run All.

In [ ]:
# --- 1. Repo + Gorev E checkpointleri ---
import subprocess, sys, os, re
os.chdir('/kaggle/working')
if not os.path.isdir('HFP'):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git'], check=True)
else:
    subprocess.run(['git','-C','HFP','pull'], check=True)
os.chdir('/kaggle/working/HFP')
import torch
assert torch.cuda.is_available(), 'GPU secili degil (T4)'
print('GPU:', torch.cuda.get_device_name(0))
assert os.path.exists('review_scripts/matched_probe.py'), 'matched_probe.py yok -> `git push` yapin'
have = [f for f in os.listdir('checkpoints') if f.startswith('carryv1')] if os.path.isdir('checkpoints') else []
print('Gorev E checkpointleri:', have or 'YOK')
ENV = dict(os.environ, PYTHONPATH='/kaggle/working/HFP')

In [ ]:
# --- 2. Gorev E checkpointleri yoksa once onlari uret (egitim) ---
# Ayni oturumda kosmadiysaniz checkpointler yoktur; bu hucre eksikleri tamamlar.
need = [(m, s) for m in ['exp','cubic_flux_chunked'] for s in ['0','1','2']
        if not os.path.exists(f'checkpoints/carryv1_{m}_s{s}.pt')]
if need:
    print('eksik kollar egitiliyor (Gorev E):', need, flush=True)
    for m, s in need:
        print(f'\n--- egitim: {m} seed {s} ---', flush=True)
        subprocess.run([sys.executable,'review_scripts/carry_curriculum.py', m, s, '1800'],
                       env=dict(ENV, CC_CARRY_MAX='16', CC_STEPS='1200'), check=True)
else:
    print('tum checkpointler mevcut, dogrudan sondaya geciliyor.')

In [ ]:
# --- 3. ESLESMIS vs ESKI sonda (egitim yok) ---
raw = {}
for m in ['exp', 'cubic_flux_chunked']:
    for s in ['0', '1', '2']:
        print(f'\n===== {m} | seed {s} =====', flush=True)
        r = subprocess.run([sys.executable, 'review_scripts/matched_probe.py', m, s],
                           env=dict(ENV, MP_KS='0,1,2,4,8,16,32,64', MP_TRIALS='30'),
                           check=True, capture_output=True, text=True)
        print(r.stdout.strip(), flush=True)
        for line in r.stdout.splitlines():
            mm = re.match(r'\s*(\d+)\s+\d+\s+\|\s+([\d.]+)%\s+\S+\s+\|\s+([\d.]+)%', line)
            if mm:
                raw[(m, s, int(mm.group(1)))] = (float(mm.group(2)), float(mm.group(3)))
print('\nham:', raw)

In [ ]:
# --- 4. OZET + on-kayitli okuma ---
import statistics as st
KS = [0,1,2,4,8,16,32,64]
print(f'{"K":>4} {"~token":>8} | {"ESLESMIS":>9} | {"ESKI":>7}   (seed-ortalama, sans %3.3)')
print('-'*52)
summ = {}
for K in KS:
    mv = [raw[(m,s,K)][0] for m in ['exp','cubic_flux_chunked'] for s in ['0','1','2'] if (m,s,K) in raw]
    uv = [raw[(m,s,K)][1] for m in ['exp','cubic_flux_chunked'] for s in ['0','1','2'] if (m,s,K) in raw]
    if mv:
        summ[K] = (st.mean(mv), st.mean(uv))
        print(f'{K:>4} {K*256:>8} | {summ[K][0]:>8.1f}% | {summ[K][1]:>6.1f}%')

trained = [summ[K][0] for K in (8, 16) if K in summ]      # egitilen menzil ici
old_tr  = [summ[K][1] for K in (8, 16) if K in summ]
if trained and min(trained) >= 15 and max(old_tr) <= 8:
    print('\nOKUMA: UYUMSUZLUK DOGRULANDI -> §17-§19 cokusu buyuk olcude EVAL ARTEFAKTI.')
    print('  Durust iddia: "egitildigi mesafede, egitildigi bicimde soruldugunda hatirlar".')
elif trained and max(trained) < 8:
    print('\nOKUMA: MIMARI SUPHESI -> dort eleme tamamlandi (yasa/kapasite/mufredat/harness).')
    print('  Sonraki eksen: state boyutu (bulk_dim) ve okuma yolu.')
else:
    print('\nOKUMA: Ara bolge — eslesmis sonda iyilestiriyor ama esik altinda; TRIALS artir.')
if 32 in summ and 16 in summ:
    print(f'  Ekstrapolasyon: K=16 {summ[16][0]:.1f}% -> K=32 {summ[32][0]:.1f}%'
          + (f' -> K=64 {summ[64][0]:.1f}%' if 64 in summ else ''))
print('\nBu ciktiyi Claude\'a yapistirin; RESULTS §20 olarak islenecek.')